## Сессия 4 - Объединенин таблиц

### Этап 1 - загрузка таблиц из CSV-файлов

In [2]:
import pandas as pd 

df_orders = pd.read_csv('data/orders.csv', sep = ';') 
df_customers = pd.read_csv('data/customers.csv', sep = ';') 
df_payments = pd.read_csv('data/payments.csv', sep = ';')
df_items = pd.read_csv('data/items.csv', sep = ';')
df_products = pd.read_csv('data/products.csv', sep = ';')
df_reviews = pd.read_csv('data/reviews.csv', sep = ';')
df_sellers = pd.read_csv('data/sellers.csv', sep = ';')
df_geolocation = pd.read_csv('data/geolocation.csv', sep = ';')




### Этап 2 - Очистка и подготовка Orders

In [3]:
df_orders['is_delivered'] = df_orders['order_delivered_customer_date'].notna()
print(df_orders['is_delivered'].value_counts())
print(f'Исключено из датасета: {df_orders['is_delivered'].value_counts(normalize = True)[False] * 100.:2f}%. Данные исключены в связи с тем, что данные заказы не были когда либо доставлены')
print()
orders = df_orders[df_orders['is_delivered']].copy().dropna()
columns = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in columns:
    orders[col] = pd.to_datetime(orders[col])
print(orders.info())

is_delivered
True     96476
False     2965
Name: count, dtype: int64
Исключено из датасета: 2.981668%. Данные исключены в связи с тем, что данные заказы не были когда либо доставлены

<class 'pandas.core.frame.DataFrame'>
Index: 96461 entries, 0 to 99440
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96461 non-null  object        
 1   customer_id                    96461 non-null  object        
 2   order_status                   96461 non-null  object        
 3   order_purchase_timestamp       96461 non-null  datetime64[ns]
 4   order_approved_at              96461 non-null  datetime64[ns]
 5   order_delivered_carrier_date   96461 non-null  datetime64[ns]
 6   order_delivered_customer_date  96461 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96461 non-null  datetime64[ns]
 8   is_delivered                   96461 n

### Этап 3 - Очистка и подготовка Customers

In [4]:
## Повешен флаг для подтверждения только тех пользователей, имеющих корректные учетнные данные
df_customers['confirm'] = (df_customers['customer_id'].notna()) & (df_customers['customer_unique_id'].notna())
## Заполнение возмодных нулевых значений на 'unknown'
df_customers[['customer_city', 'customer_state']] = (
    df_customers[['customer_city', 'customer_state']].fillna('unknown')
)
print(f"Всего подтвержденных пользователей: {len(df_customers[df_customers['confirm']])}")
print()
counts = len(df_customers) - df_customers['confirm'].sum()
print(f"Всего пользователей vs подтвержденных: {len(df_customers)} / {df_customers['confirm'].sum()}, разница: {counts}")
print()
if counts > 0:
    print('Необходима очистка от нулевых значений')
    customers = df_customers[df_customers['conirm']].copy()
    print(customers.info())
else:
    print('Нулевые значения отсутствуют')
    customers = df_customers.copy()
    print(customers.info())

Всего подтвержденных пользователей: 99441

Всего пользователей vs подтвержденных: 99441 / 99441, разница: 0

Нулевые значения отсутствуют
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 6 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
 5   confirm                   99441 non-null  bool  
dtypes: bool(1), int64(1), object(4)
memory usage: 3.9+ MB
None


### Этап 4 - Очистка и подготовка payments

In [ ]:
## Вешаем флаг и убираем возможные нулевые значения в полях
df_payments['conf_&_paid'] = (df_payments['order_id'].notna()) & (df_payments['payment_sequential'].notna()) & (df_payments['payment_value'].ne(0)) & (df_payments['payment_value'].notna()) 
df_payments['payment_type'] = df_payments['payment_type'].fillna('unknown')
counts = len(df_payments) - df_payments['conf_&_paid'].sum()
print(f"Неподтвержденные vs подтвержденные: {len(df_payments)} / {df_payments['conf_&_paid'].sum()}, разница: {counts}")
print()
if counts > 0:
    print('Необходима очистка от нулевых значений')
    payments = df_payments[df_payments['conf_&_paid']].copy()
    print(payments.info())
else:
    print('Нулевые значения отсутствуют')
    payments = df_payments.copy()
    print(customers.info())

Неподтвержденные vs подтвержденные: 103886 / 103877, разница: 9

Необходима очистка от нулевых значений
<class 'pandas.core.frame.DataFrame'>
Index: 103877 entries, 0 to 103885
Data columns (total 6 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103877 non-null  object 
 1   payment_sequential    103877 non-null  int64  
 2   payment_type          103877 non-null  object 
 3   payment_installments  103877 non-null  int64  
 4   payment_value         103877 non-null  float64
 5   conf_&_paid           103877 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(2)
memory usage: 4.9+ MB
None


### Этап 5 - Подготовка Items

In [6]:
print(df_items.info())
print(df_items.isnull().sum())
print(df_items.duplicated().sum())
print(f"Есть ли дубликаты: {df_items.duplicated(subset = ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date'])}")
print(df_items.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB
None
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64
0
Есть ли дубликаты: 0         False
1         False
2         False
3         False
4         False
          ...  
112645    False
112646    False
112647    False
112648    F

### Этап 6 - Подготовка products 

In [7]:
print(df_products.info())
print(df_products.isnull().sum())
print(df_products.duplicated().sum())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB
None
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
prod

In [8]:
df_products['product_category_name'] = (df_products['product_category_name'].fillna('unknown'))
nan_cols = ['product_description_lenght', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
df_products[nan_cols] = df_products[nan_cols].fillna(df_products[nan_cols].median())
df_products['product_photos_qty'] = df_products['product_photos_qty'].fillna(0)
print(df_products.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32951 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32951 non-null  float64
 4   product_photos_qty          32951 non-null  float64
 5   product_weight_g            32951 non-null  float64
 6   product_length_cm           32951 non-null  float64
 7   product_height_cm           32951 non-null  float64
 8   product_width_cm            32951 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB
None


### Этап 7 - Подготовка Reviews

In [9]:
print(df_reviews.info())
print(df_reviews.isnull().sum())
print(df_reviews.duplicated().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77920 entries, 0 to 77919
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   review_id                77918 non-null  object 
 1   order_id                 77917 non-null  object 
 2   review_score             77917 non-null  float64
 3   review_comment_title     9204 non-null   object 
 4   review_comment_message   32235 non-null  object 
 5   review_creation_date     77916 non-null  object 
 6   review_answer_timestamp  77916 non-null  object 
dtypes: float64(1), object(6)
memory usage: 4.2+ MB
None
review_id                      2
order_id                       3
review_score                   3
review_comment_title       68716
review_comment_message     45685
review_creation_date           4
review_answer_timestamp        4
dtype: int64
1


In [10]:
print(df_reviews[df_reviews.duplicated(keep = False)])

      review_id order_id  review_score review_comment_title  \
77917       NaN      NaN           NaN                  NaN   
77919       NaN      NaN           NaN                  NaN   

      review_comment_message review_creation_date review_answer_timestamp  
77917                    NaN                  NaN                     NaN  
77919                    NaN                  NaN                     NaN  


In [11]:
df_reviews = df_reviews.drop_duplicates()
df_reviews.info()


<class 'pandas.core.frame.DataFrame'>
Index: 77919 entries, 0 to 77918
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   review_id                77918 non-null  object 
 1   order_id                 77917 non-null  object 
 2   review_score             77917 non-null  float64
 3   review_comment_title     9204 non-null   object 
 4   review_comment_message   32235 non-null  object 
 5   review_creation_date     77916 non-null  object 
 6   review_answer_timestamp  77916 non-null  object 
dtypes: float64(1), object(6)
memory usage: 4.8+ MB


In [ ]:
print(df_reviews['order_id'].isna().sum())
print(f"Уникальных заказов: {df_reviews['order_id'].nunique()}")
review_count = df_reviews['order_id'].value_counts()
df_reviews = df_reviews.dropna(subset = ['order_id'])
df_reviews.isna().sum()
cols = ['review_creation_date', 'review_answer_timestamp']
for col in cols:
    df_reviews[col] = pd.to_datetime(df_reviews[col])

string_cols = ['review_comment_title', 'review_comment_message']
df_reviews[string_cols] = df_reviews[string_cols].fillna('No message')

df_reviews[cols] = df_reviews[cols].fillna(df_reviews[cols].median())
print('final info')
print()
df_reviews.info()

0
Уникальных заказов: 77576
final info

<class 'pandas.core.frame.DataFrame'>
Index: 77917 entries, 0 to 77916
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                77917 non-null  object        
 1   order_id                 77917 non-null  object        
 2   review_score             77917 non-null  float64       
 3   review_comment_title     77917 non-null  object        
 4   review_comment_message   77917 non-null  object        
 5   review_creation_date     77917 non-null  datetime64[ns]
 6   review_answer_timestamp  77917 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(1), object(4)
memory usage: 4.8+ MB


В процессе подготовки датасета были обнаружены и исправлены следующие проблемы:
- 1. Наличие большого количества дубликатов

## Этап 8 - Подготовка Sellers

In [22]:

df_sellers.rename(columns = {'seller_zip_code_prefix': 'zip_code_prefix'}, inplace = True)
df_sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   seller_id        3095 non-null   object
 1   zip_code_prefix  3095 non-null   int64 
 2   seller_city      3095 non-null   object
 3   seller_state     3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


**Датафрейм чист и не нуждается в дополнительной подготовке**

## Этап 9 - Подготовка Geolocation

In [25]:

df_geolocation.rename(columns = {'geolocation_zip_code_prefix': 'zip_code_prefix'}, inplace = True)
df_geolocation = df_geolocation.groupby('zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean',
    'geolocation_city': 'first',
    'geolocation_state': 'first'
}).reset_index()

df_geolocation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19015 entries, 0 to 19014
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   zip_code_prefix    19015 non-null  int64  
 1   geolocation_lat    19015 non-null  float64
 2   geolocation_lng    19015 non-null  float64
 3   geolocation_city   19015 non-null  object 
 4   geolocation_state  19015 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 742.9+ KB


**Датафрейм чист и не нуждается в подготовке**

### Этап 10 - Соединение таблиц

In [29]:
df_full = pd.merge(
    df_orders,
    df_customers,
    on = 'customer_id',
    how = 'inner'
)
df_full = pd.merge(
    df_full,
    df_payments,
    on = 'order_id',
    how = 'inner'
)
df_full = pd.merge(
    df_full,
    df_items,
    on = 'order_id',
    how = 'inner'
)
df_full = pd.merge(
    df_full,
    df_reviews,
    on = 'order_id',
    how = 'inner'
)
df_full = pd.merge(
    df_full,
    df_products,
    on = 'product_id',
    how = 'inner'
)
df_full = pd.merge(
    df_full,
    df_sellers,
    on = 'seller_id',
    how = 'left'
)
df_full = pd.merge(
    df_full,
    df_geolocation,
    on = 'zip_code_prefix',
    how = 'left'
)

df_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 92247 entries, 0 to 92246
Data columns (total 46 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       92247 non-null  object        
 1   customer_id                    92247 non-null  object        
 2   order_status                   92247 non-null  object        
 3   order_purchase_timestamp       92247 non-null  object        
 4   order_approved_at              92235 non-null  object        
 5   order_delivered_carrier_date   91250 non-null  object        
 6   order_delivered_customer_date  90271 non-null  object        
 7   order_estimated_delivery_date  92247 non-null  object        
 8   is_delivered                   92247 non-null  bool          
 9   customer_unique_id             92247 non-null  object        
 10  customer_zip_code_prefix       92247 non-null  int64         
 11  customer_city  

В процессе объединения таблиц произошла потеря ~4,5% данных. Данная потеря незначительно повлияет на дальнейший анализ и является ожидаемой при использовании Inner Join. 

**Обоснование:** 
- Заказы, в которых отсутствуют ключевые поля не могут быть корректно учтены при рассчете метрик, что приведет к искажению
- Заказы без ключевых полей не могут быть использованы при категориальном анализе

